# 01: Weather data pipeline

Steps 1 to 4 of Module 1. Run the cells top to bottom. Every cell is self-contained, so this
notebook works in Google Colab with no repository checkout.

What you get at the end:
- a locations table (step 1)
- daily weather history from Open-Meteo and NASA POWER (steps 2 and 3)
- a forecast (step 4)
- a comparison of the two history sources, and an ET0 check against Open-Meteo

In [ ]:
# Install anything Colab might be missing (usually nothing)
!pip -q install requests pandas numpy matplotlib

## Step 1: Locations

In [ ]:
import io
import pandas as pd

LOCATIONS_CSV = '''name,state,lat,lon,elevation_m
Latur,Maharashtra,18.4088,76.5604,636
Nagpur,Maharashtra,21.1458,79.0882,310
Pune,Maharashtra,18.5204,73.8567,560
Solapur,Maharashtra,17.6599,75.9064,458
Delhi,Delhi,28.6139,77.2090,216
'''
locations = pd.read_csv(io.StringIO(LOCATIONS_CSV)).set_index('name')
locations

## Helpers (HTTP, Open-Meteo, NASA POWER, ET0)

In [ ]:
"""Shared helpers: HTTP with retries, raw-response saving."""
import json
import os
import time

import requests


def http_get_json(url, params, retries=3, timeout=60, raw_path=None):
    """GET a JSON API with simple retry/backoff. Optionally save the raw response to disk."""
    last_err = None
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            data = r.json()
            if raw_path:
                os.makedirs(os.path.dirname(raw_path), exist_ok=True)
                with open(raw_path, "w") as f:
                    json.dump(data, f)
            return data
        except Exception as e:  # network error, 429, bad JSON
            last_err = e
            time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"Request failed after {retries} tries: {url} ({last_err})")

In [ ]:
"""FAO-56 Penman-Monteith reference evapotranspiration (ET0), daily time step.

Reference: Allen et al. (1998), FAO Irrigation and Drainage Paper 56.
All inputs may be floats or numpy arrays / pandas Series.
"""
import numpy as np
import pandas as pd


def sat_vapour_pressure(t_c):
    """Saturation vapour pressure (kPa) at air temperature t_c (deg C). FAO-56 eq. 11."""
    return 0.6108 * np.exp(17.27 * t_c / (t_c + 237.3))


def extraterrestrial_radiation(lat_deg, doy):
    """Extraterrestrial radiation Ra (MJ m-2 day-1). FAO-56 eq. 21."""
    phi = np.radians(lat_deg)
    dr = 1 + 0.033 * np.cos(2 * np.pi * doy / 365)
    delta = 0.409 * np.sin(2 * np.pi * doy / 365 - 1.39)
    ws = np.arccos(np.clip(-np.tan(phi) * np.tan(delta), -1, 1))
    return (24 * 60 / np.pi) * 0.0820 * dr * (
        ws * np.sin(phi) * np.sin(delta) + np.cos(phi) * np.cos(delta) * np.sin(ws)
    )


def wind_10m_to_2m(u10):
    """Convert wind speed measured at 10 m to 2 m. FAO-56 eq. 47."""
    return u10 * 4.87 / np.log(67.8 * 10 - 5.42)


def et0_penman_monteith(tmax, tmin, rh_mean, u2, rs, lat_deg, elev_m, doy, ea=None):
    """Daily ET0 in mm/day.

    tmax, tmin : deg C
    rh_mean    : mean relative humidity, percent
    u2         : wind speed at 2 m height, m/s
    rs         : incoming solar radiation, MJ m-2 day-1
    lat_deg    : latitude in degrees (north positive)
    elev_m     : elevation above sea level, m
    doy        : day of year (1-366)
    ea         : optional actual vapour pressure (kPa); if None it is derived from rh_mean
    """
    tmean = (tmax + tmin) / 2
    pressure = 101.3 * ((293 - 0.0065 * elev_m) / 293) ** 5.26
    gamma = 0.000665 * pressure
    delta = 4098 * sat_vapour_pressure(tmean) / (tmean + 237.3) ** 2

    es = (sat_vapour_pressure(tmax) + sat_vapour_pressure(tmin)) / 2
    if ea is None:
        ea = rh_mean / 100 * es

    ra = extraterrestrial_radiation(lat_deg, doy)
    rso = (0.75 + 2e-5 * elev_m) * ra
    rs_rso = np.clip(rs / rso, 0.3, 1.0)
    rns = (1 - 0.23) * rs
    rnl = 4.903e-9 * (((tmax + 273.16) ** 4 + (tmin + 273.16) ** 4) / 2) * (
        0.34 - 0.14 * np.sqrt(ea)
    ) * (1.35 * rs_rso - 0.35)
    rn = rns - rnl

    num = 0.408 * delta * rn + gamma * 900 / (tmean + 273) * u2 * (es - ea)
    den = delta + gamma * (1 + 0.34 * u2)
    return num / den


def add_et0(df, lat_deg, elev_m, col="et0_calc_mm"):
    """Add a computed ET0 column to a harmonised daily weather DataFrame.

    Needs columns: date, tmax, tmin, rh_mean, wind_2m_ms, solar_mj.
    """
    out = df.copy()
    doy = pd.to_datetime(out["date"]).dt.dayofyear.to_numpy()
    out[col] = et0_penman_monteith(
        out["tmax"].to_numpy(float), out["tmin"].to_numpy(float),
        out["rh_mean"].to_numpy(float), out["wind_2m_ms"].to_numpy(float),
        out["solar_mj"].to_numpy(float), lat_deg, elev_m, doy,
    )
    return out

In [ ]:
import pandas as pd  # used by the modules below
"""Open-Meteo fetchers (history + forecast), returning the project's harmonised daily columns.

Harmonised columns: date, tmax, tmin, rain_mm, solar_mj, rh_mean, wind_2m_ms, et0_openmeteo_mm
No API key needed. Free for non-commercial use.
"""
import pandas as pd


ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
FORECAST_URL = "https://api.open-meteo.com/v1/forecast"

DAILY_VARS = ["temperature_2m_max", "temperature_2m_min", "precipitation_sum",
              "shortwave_radiation_sum", "et0_fao_evapotranspiration"]
HOURLY_VARS = ["relative_humidity_2m", "wind_speed_10m"]


def _wind_10m_to_2m(u10):
    import numpy as np
    return u10 * 4.87 / np.log(67.8 * 10 - 5.42)


def _parse(data):
    daily = pd.DataFrame(data["daily"]).rename(columns={
        "time": "date", "temperature_2m_max": "tmax", "temperature_2m_min": "tmin",
        "precipitation_sum": "rain_mm", "shortwave_radiation_sum": "solar_mj",
        "et0_fao_evapotranspiration": "et0_openmeteo_mm"})
    hourly = pd.DataFrame(data["hourly"])
    hourly["date"] = hourly["time"].str.slice(0, 10)
    agg = hourly.groupby("date").agg(
        rh_mean=("relative_humidity_2m", "mean"),
        wind_10m=("wind_speed_10m", "mean")).reset_index()
    agg["wind_2m_ms"] = _wind_10m_to_2m(agg["wind_10m"])
    out = daily.merge(agg[["date", "rh_mean", "wind_2m_ms"]], on="date", how="left")
    out["date"] = pd.to_datetime(out["date"])
    out.attrs["elevation_m"] = data.get("elevation")
    return out[["date", "tmax", "tmin", "rain_mm", "solar_mj", "rh_mean",
                "wind_2m_ms", "et0_openmeteo_mm"]]


def fetch_history(lat, lon, start, end, raw_path=None):
    """Daily weather history. start/end as 'YYYY-MM-DD'."""
    params = {"latitude": lat, "longitude": lon, "start_date": start, "end_date": end,
              "daily": ",".join(DAILY_VARS), "hourly": ",".join(HOURLY_VARS),
              "wind_speed_unit": "ms", "timezone": "Asia/Kolkata"}
    return _parse(http_get_json(ARCHIVE_URL, params, raw_path=raw_path))


def fetch_forecast(lat, lon, days=16, past_days=0, raw_path=None):
    """Daily forecast for the next `days` (max 16), optionally with some recent past days."""
    params = {"latitude": lat, "longitude": lon, "forecast_days": days,
              "past_days": past_days, "daily": ",".join(DAILY_VARS),
              "hourly": ",".join(HOURLY_VARS), "wind_speed_unit": "ms",
              "timezone": "Asia/Kolkata"}
    return _parse(http_get_json(FORECAST_URL, params, raw_path=raw_path))

In [ ]:
"""NASA POWER daily fetcher, returning the same harmonised columns as the Open-Meteo module
(minus ET0, which POWER does not provide).

Harmonised columns: date, tmax, tmin, rain_mm, solar_mj, rh_mean, wind_2m_ms
No API key needed. POWER data lags real time by a few days.
"""
import pandas as pd


POWER_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"
PARAMS = ["T2M_MAX", "T2M_MIN", "PRECTOTCORR", "RH2M", "WS2M", "ALLSKY_SFC_SW_DWN"]


def fetch_history_power(lat, lon, start, end, raw_path=None):
    """Daily history. start/end as 'YYYY-MM-DD'."""
    params = {"parameters": ",".join(PARAMS), "community": "AG",
              "latitude": lat, "longitude": lon,
              "start": start.replace("-", ""), "end": end.replace("-", ""),
              "format": "JSON"}
    data = http_get_json(POWER_URL, params, raw_path=raw_path)
    p = data["properties"]["parameter"]
    df = pd.DataFrame({k: pd.Series(v) for k, v in p.items()})
    df.index.name = "date"
    df = df.reset_index()
    df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")
    df = df.replace(-999, float("nan"))  # POWER's missing-data flag
    df = df.rename(columns={"T2M_MAX": "tmax", "T2M_MIN": "tmin", "PRECTOTCORR": "rain_mm",
                            "RH2M": "rh_mean", "WS2M": "wind_2m_ms",
                            "ALLSKY_SFC_SW_DWN": "solar_mj"})
    return df[["date", "tmax", "tmin", "rain_mm", "solar_mj", "rh_mean", "wind_2m_ms"]]

## Steps 2 and 3: One test case, both sources

Soybean in Latur, kharif season 2023 (1 June to 31 October). One district and one season keeps
everything checkable. Raw responses are saved to `data/raw/` so you never re-download them.

In [ ]:
city = 'Latur'
lat, lon, elev = locations.loc[city, ['lat', 'lon', 'elevation_m']]
start, end = '2023-06-01', '2023-10-31'

om = fetch_history(lat, lon, start, end, raw_path=f'data/raw/openmeteo_{city}_{start}_{end}.json')
pw = fetch_history_power(lat, lon, start, end, raw_path=f'data/raw/nasapower_{city}_{start}_{end}.json')

import os
os.makedirs('data/processed', exist_ok=True)
om.to_csv(f'data/processed/openmeteo_{city}_2023_kharif.csv', index=False)
pw.to_csv(f'data/processed/nasapower_{city}_2023_kharif.csv', index=False)
print(len(om), 'Open-Meteo days;', len(pw), 'NASA POWER days')
om.head()

## Compare the two sources

Where the free sources disagree at the same point is early evidence for the project's research question.

In [ ]:
import matplotlib.pyplot as plt

m = om.merge(pw, on='date', suffixes=('_om', '_pw'))
print('Total rain (mm):  Open-Meteo %.0f   NASA POWER %.0f' % (m.rain_mm_om.sum(), m.rain_mm_pw.sum()))
print('Mean Tmax (C):    Open-Meteo %.1f   NASA POWER %.1f' % (m.tmax_om.mean(), m.tmax_pw.mean()))
print('Daily rain correlation: %.2f' % m.rain_mm_om.corr(m.rain_mm_pw))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(m.date, m.rain_mm_om.cumsum(), label='Open-Meteo')
ax[0].plot(m.date, m.rain_mm_pw.cumsum(), label='NASA POWER')
ax[0].set_title(f'{city}: cumulative rain (mm)'); ax[0].legend()
ax[1].plot(m.date, m.tmax_om, label='Open-Meteo')
ax[1].plot(m.date, m.tmax_pw, label='NASA POWER')
ax[1].set_title(f'{city}: daily max temperature (C)'); ax[1].legend()
plt.tight_layout(); plt.show()

## ET0 check

Compute FAO-56 ET0 ourselves from the raw weather variables and compare with the value Open-Meteo
publishes. If they agree closely, the calculation can be trusted as the base of the water-balance engine.

In [ ]:
mine = add_et0(om, lat, elev)
d = mine.dropna(subset=['et0_calc_mm', 'et0_openmeteo_mm'])
err = d.et0_calc_mm - d.et0_openmeteo_mm
print('Mean ET0  mine: %.2f   Open-Meteo: %.2f  (mm/day)' % (d.et0_calc_mm.mean(), d.et0_openmeteo_mm.mean()))
print('Mean error %.2f mm/day,  RMSE %.2f mm/day,  correlation %.2f' % (err.mean(), (err**2).mean()**0.5, d.et0_calc_mm.corr(d.et0_openmeteo_mm)))
plt.figure(figsize=(5, 5))
plt.scatter(d.et0_openmeteo_mm, d.et0_calc_mm, s=8)
lim = [0, max(d.et0_openmeteo_mm.max(), d.et0_calc_mm.max()) + 0.5]
plt.plot(lim, lim, 'k--'); plt.xlim(lim); plt.ylim(lim)
plt.xlabel('Open-Meteo ET0 (mm/day)'); plt.ylabel('My ET0 (mm/day)'); plt.show()

## Step 4: Forecast

Next 7 days for the same location, plus the last 3 days for context. The in-season engine will use
this for decisions like "rain is coming, do not irrigate today".

In [ ]:
fc = fetch_forecast(lat, lon, days=7, past_days=3, raw_path=f'data/raw/openmeteo_forecast_{city}.json')
fc[['date', 'tmax', 'tmin', 'rain_mm', 'et0_openmeteo_mm']]

## What next

Step 5 onwards: IMD gridded rainfall, soil from SoilGrids, then the FAO-56 water balance for the soybean season.